# Stream on Colab (T4 GPU)
Byte-level SSM language model â€” token-free, position-free, O(n).

**What this notebook does:**
1. Clones the repo
2. Installs dependencies (inc. `torch.compile` + Triton support)
3. Downloads TinyStories â†’ byte-level dataset
4. Verifies model loads + runs
5. Benchmarks Stream vs GPT on GPU
6. Runs scaling curve (4 sizes)
7. Runs training (configurable)
8. Saves results to Google Drive

**Runtime:** Runtime â†’ Change runtime type â†’ T4 GPU

In [ ]:
# @title 1. Mount Drive & Clone Repo
import os, sys

# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/nishantXnova/RETRANS-X.git'
PROJECT_DIR = '/content/RETRANS-X'
DRIVE_DIR = '/content/drive/MyDrive/stream_results'

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print('Repo already cloned, pulling latest...')
    %cd {PROJECT_DIR}
    !git pull

%cd {PROJECT_DIR}
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'\nProject at: {PROJECT_DIR}')
print(f'Results at: {DRIVE_DIR}')

In [ ]:
# @title 2. Install Dependencies
import torch
print(f'PyTorch {torch.__version__}, CUDA {torch.version.cuda}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'Compute Capability: {torch.cuda.get_device_capability(0)}')

# Check if triton is available
try:
    import triton
    print(f'Triton {triton.__version__} available â€” torch.compile will work!')
except:
    print('Installing triton...')
    !pip install -q triton
    import triton
    print(f'Triton {triton.__version__} installed')

# Verify torch.compile works
def test_fn(x): return x * 2
compiled = torch.compile(test_fn)
_ = compiled(torch.tensor([1.0], device='cuda'))
print('torch.compile() works!')

# Try installing mamba-ssm (optional â€” may have build issues)
try:
    !pip install -q mamba-ssm 2>&1 | tail -5
    from mamba_ssm import Mamba
    print('mamba-ssm installed!')
except:
    print('mamba-ssm not installed (optional, not needed for Stream)')

In [ ]:
# @title 3. Prepare Data
import sys, os

# Fallback if kernel restarted
if 'PROJECT_DIR' not in globals():
    PROJECT_DIR = '/content/RETRANS-X'

os.chdir(PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, 'VECTOR'))

from data.prepare_bytes import prepare_bytes

DATA_DIR = 'VECTOR/data/bytes'
TRAIN_FILE = os.path.join(DATA_DIR, 'train.bin')
VAL_FILE = os.path.join(DATA_DIR, 'val.bin')

if not os.path.exists(TRAIN_FILE):
    print('Creating byte-level dataset...')
    os.makedirs(DATA_DIR, exist_ok=True)
    tinystories_url = 'https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStories-train.txt'
    tinystories_path = os.path.join(DATA_DIR, 'TinyStories-train.txt')

    if not os.path.exists(tinystories_path):
        print('Downloading TinyStories (940MB)...')
        import requests
        r = requests.get(tinystories_url, stream=True)
        with open(tinystories_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk:
                    f.write(chunk)
        print('Download complete.')

    print('Processing {} into binary format...'.format(tinystories_path))
    print('(reads file as raw bytes, no full-string decode - RAM safe)')
    prepare_bytes(tinystories_path)
else:
    print('Data already prepared.')

# Safe file size check
if os.path.exists(TRAIN_FILE) and os.path.exists(VAL_FILE):
    train_size = os.path.getsize(TRAIN_FILE)
    val_size = os.path.getsize(VAL_FILE)
    print('Dataset ready: Train: {:.1f} MB, Val: {:.1f} MB'.format(train_size/1e6, val_size/1e6))
else:
    print('Dataset files not found. Check if prepare_bytes ran correctly.')


In [ ]:
# @title 4. Verify Model Loads
sys.path.insert(0, PROJECT_DIR + '/VECTOR')
from model import Stream, StreamConfig

device = 'cuda'
model = Stream(StreamConfig(
    n_embd=128, n_layer=2, ssm_d_state=8,
    n_predict=4, block_size=256
)).to(device)

x = torch.randint(0, 256, (1, 256), device=device)
logits, loss = model(x, targets=x)
print(f'Forward OK, loss = {loss.item():.4f}')

loss.backward()
print('Backward OK')
print(f'Parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M')

# Test torch.compile on full model
print('\nCompiling model with torch.compile...')
model_compiled = torch.compile(model)
x = torch.randint(0, 256, (1, 256), device=device)
logits, loss = model_compiled(x, targets=x)
print(f'Compiled forward OK, loss = {loss.item():.4f}')
loss.backward()
print('Compiled backward OK')

In [ ]:
# @title 4b. Fused Triton SSM Scan (check + benchmark)
import sys, os, time, numpy as np, torch

import importlib.util

# === Self-contained path setup (no dependencies on other cells) ===
vec_dir = None
for d in [os.path.join(os.getcwd(), 'VECTOR'), '/content/RETRANS-X/VECTOR']:
    if os.path.isdir(d):
        vec_dir = d
        break

if vec_dir is None:
    # Try one level up (if CWD is inside VECTOR)
    parent = os.path.dirname(os.getcwd())
    if os.path.isdir(os.path.join(parent, 'VECTOR')):
        vec_dir = os.path.join(parent, 'VECTOR')

if vec_dir is None:
    print('ERROR: Cannot find VECTOR/ directory')
    print(f'  CWD: {os.getcwd()}')
    print(f'  Contents: {os.listdir(".")}')
    print('Run cell 1 first (clone repo), then restart runtime and re-run')
    raise SystemExit(1)

sys.path.insert(0, vec_dir)
print(f'VECTOR dir: {vec_dir}')
triton_path = os.path.join(vec_dir, 'triton_scan.py')
print(f'triton_scan.py exists: {os.path.isfile(triton_path)}')

if not os.path.isfile(triton_path):
    print('triton_scan.py not found! Running git pull...')
    os.chdir(os.path.dirname(vec_dir))
    !git pull 2>&1 | tail -5
    if not os.path.isfile(triton_path):
        print('STILL not found.', os.listdir(vec_dir))
        raise SystemExit(1)

# Load module directly from file (bypasses sys.path / caching issues)
spec = importlib.util.spec_from_file_location('triton_scan', triton_path)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
triton_ssm_scan = mod.triton_ssm_scan
HAS_TRITON = mod.HAS_TRITON
enable_triton = mod.enable_triton

if not HAS_TRITON:
    print('Triton NOT available on this runtime.')
    print('Expected on Colab T4 with Linux. Check cell 2 for details.')
else:
    print('Triton available! Running scan correctness check...')
    device = 'cuda'
    T, H, N, B = 2048, 128, 8, 2
    a = torch.randn(B, T, H, N, device=device)
    b = torch.randn(B, T, H, N, device=device)

    # Triton forward
    h_tri = triton_ssm_scan(a, b, T)

    # Reference: pure PyTorch loop (no JIT dependency)
    h_ref = torch.zeros(B, H, N, device=device)
    h_ref_out = torch.empty_like(a)
    for t in range(T):
        h_ref = h_ref * a[:, t] + b[:, t]
        h_ref_out[:, t] = h_ref

    max_diff = (h_ref_out - h_tri).abs().max().item()
    print(f'  Max diff (fwd vs reference): {max_diff:.2e}')
    assert max_diff < 1e-3, f'Forward mismatch too large: {max_diff:.2e}'
    print('  Forward: CORRECT')

    # Speed benchmark
    print('\nSpeed benchmark (T=4096, H=128, N=8, B=4, 20 iters)...')
    a_big = torch.randn(4, 4096, 128, 8, device=device)
    b_big = torch.randn(4, 4096, 128, 8, device=device)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(20):
        _ = triton_ssm_scan(a_big, b_big, 4096)
    torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) / 20 * 1000
    print(f'  Triton: {ms:.2f} ms per scan')

    # JIT reference benchmark
    jit_path = os.path.join(vec_dir, 'model.py')
    if os.path.isfile(jit_path):
        try:
            jit_spec = importlib.util.spec_from_file_location('jit_bench', jit_path)
            mod_jit = importlib.util.module_from_spec(jit_spec)
            jit_spec.loader.exec_module(mod_jit)
            jit_fn = mod_jit._ssm_scan
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            for _ in range(20):
                _ = jit_fn(a_big, b_big, 4096)
            torch.cuda.synchronize()
            jit_ms = (time.perf_counter() - t0) / 20 * 1000
            print(f'  JIT:     {jit_ms:.2f} ms per scan')
            print(f'  Speedup: {jit_ms / ms:.2f}x')
        except Exception as e:
            print(f'  JIT benchmark skipped: {e}')
    else:
        print('  JIT benchmark skipped: model.py not found')

    print('\nTriton scan ready for training!')


In [ ]:
# @title 4c. Full GPU Benchmark WITH Triton
# Self-contained — run after cell 4b
import sys, os, time, numpy as np, torch, importlib.util

# --- Find VECTOR directory (same as cell 4b) ---
vec_dir = None
for d in [os.path.join(os.getcwd(), 'VECTOR'), '/content/RETRANS-X/VECTOR']:
    if os.path.isdir(d):
        vec_dir = d; break
if vec_dir is None:
    p = os.path.dirname(os.getcwd())
    if os.path.isdir(os.path.join(p, 'VECTOR')): vec_dir = os.path.join(p, 'VECTOR')
assert vec_dir is not None, 'Cannot find VECTOR/ dir'

proj_dir = os.path.dirname(vec_dir)

# --- Load triton_scan module ---
ts_spec = importlib.util.spec_from_file_location('triton_scan', os.path.join(vec_dir, 'triton_scan.py'))
ts_mod = importlib.util.module_from_spec(ts_spec); ts_spec.loader.exec_module(ts_mod)
enable_triton = ts_mod.enable_triton

# --- Load Stream from model.py ---
m_spec = importlib.util.spec_from_file_location('stream_model', os.path.join(vec_dir, 'model.py'))
m_mod = importlib.util.module_from_spec(m_spec); m_spec.loader.exec_module(m_mod)
Stream = m_mod.Stream; StreamConfig = m_mod.StreamConfig

# --- Load nanoGPT ---
ngpt_path = os.path.join(proj_dir, 'nanoGPT', 'model.py')
if os.path.isfile(ngpt_path):
    ng_spec = importlib.util.spec_from_file_location('nanoGPT_mod', ngpt_path)
    ng_mod = importlib.util.module_from_spec(ng_spec); ng_spec.loader.exec_module(ng_mod)
    GPT = ng_mod.GPT; GPTConfig = ng_mod.GPTConfig
    has_gpt = True
else:
    print('nanoGPT/model.py not found — will skip GPT benchmarks')
    has_gpt = False

device = 'cuda'
TS = [512, 1024, 2048, 4096, 8192, 16384]
BS = [1, 4]

results = {}

def bench_one(label, model_fn, use_triton=False):
    model = model_fn()
    if use_triton:
        enable_triton(model, enabled=True)
    model.eval()
    print(f'--- {label} ---')
    for B in BS:
        for T in TS:
            try:
                x = torch.randint(0, 256, (B, T), device=device)
                with torch.no_grad():
                    for _ in range(5): model(x)
                    torch.cuda.synchronize()
                    n = 15 if T <= 2048 else 8
                    times = []
                    for _ in range(n):
                        torch.cuda.synchronize()
                        t0 = time.perf_counter()
                        model(x)
                        torch.cuda.synchronize()
                        times.append(time.perf_counter() - t0)
                ms = np.median(times) * 1000
                tok_s = B * T / (ms / 1000)
                print(f'  T={T:>5}, B={B}: {ms:>8.2f} ms ({tok_s:>8.0f} tok/s)')
                results[(label, T, B)] = ms
            except Exception as e:
                print(f'  T={T:>5}, B={B}: {str(e)[:80]}')

bench_one('Stream 4L/128D + Triton', lambda: Stream(StreamConfig(n_embd=128, n_layer=4, ssm_d_state=8, block_size=max(TS))).to(device), use_triton=True)
bench_one('Stream 6L/256D + Triton', lambda: Stream(StreamConfig(n_embd=256, n_layer=6, ssm_d_state=16, block_size=max(TS))).to(device), use_triton=True)

if has_gpt:
    bench_one('GPT 3L/128D', lambda: GPT(GPTConfig(vocab_size=256, n_embd=128, n_layer=3, n_head=4, block_size=max(TS), dropout=0.0, bias=False)).to(device).eval())
    bench_one('GPT 8L/192D', lambda: GPT(GPTConfig(vocab_size=256, n_embd=192, n_layer=8, n_head=6, block_size=max(TS), dropout=0.0, bias=False)).to(device).eval())

# --- Summary table ---
print('\n' + '=' * 120)
hdr = '{:>5} {:>2} {:>14} {:>14} {:>14} {:>14}'
print(hdr.format('T', 'B', 'GPT3_ms', 'GPT8_ms', 'S4+T_ms', 'S6+T_ms'))
print('-' * 120)
for T in TS:
    for B in BS:
        r = lambda n: results.get((n, T, B), None)
        names = ['GPT 3L/128D', 'GPT 8L/192D', 'Stream 4L/128D + Triton', 'Stream 6L/256D + Triton']
        vals = [r(n) for n in names]
        if any(v is not None for v in vals):
            line = '{:>5} {:>2}'.format(T, B)
            for v in vals:
                line += '{:>14.2f}'.format(v) if v is not None else '{:>14}'.format('N/A')
            print(line)

# Save
import pickle
try:
    dr = '/content/drive/MyDrive/stream_results'
    os.makedirs(dr, exist_ok=True)
    with open(os.path.join(dr, 'triton_bench_results.pkl'), 'wb') as f:
        pickle.dump(results, f)
    print(f'\nSaved to {dr}/triton_bench_results.pkl')
except:
    print('\n(Results not saved)')


In [ ]:
# @title 5. GPU Bench: Stream vs GPT (compiled & uncompiled)
import sys, time, numpy as np, torch

sys.path.insert(0, PROJECT_DIR + '/VECTOR')
from model import Stream, StreamConfig

import importlib.util
spec = importlib.util.spec_from_file_location('ngpt', 'nanoGPT/model.py')
ngpt = importlib.util.module_from_spec(spec)
sys.modules['ngpt'] = ngpt
spec.loader.exec_module(ngpt)

device = 'cuda'
TS = [512, 1024, 2048, 4096, 8192]
BS = [1, 4]

results = {}

def bench_one(name, model_fn, use_compile=False):
    model = model_fn(max(TS))
    if use_compile:
        print('Compiling {}... (lazy, pay cost on first forward)'.format(name))
        t0 = time.time()
        model = torch.compile(model, dynamic=True)
        print('  Compile returned in {:.1f}s (lazy, dynamic=True)'.format(time.time() - t0))
    # Pre-warmup at max T to trigger shape compilation once
    if use_compile:
        x0 = torch.randint(0, 256, (1, max(TS)), device=device)
        with torch.no_grad():
            t0 = time.time()
            model(x0)
            print('  Compile (first forward at T={}): {:.1f}s'.format(max(TS), time.time() - t0))
    print('--- {} ---'.format(name))
    for B in BS:
        for T in TS:
            try:
                x = torch.randint(0, 256, (B, T), device=device)
                with torch.no_grad():
                    for _ in range(5):
                        model(x)
                    torch.cuda.synchronize()
                    torch.cuda.synchronize()
                    times = []
                    n = 15 if T <= 2048 else 8
                    for _ in range(n):
                        torch.cuda.synchronize()
                        t0 = time.perf_counter()
                        model(x)
                        torch.cuda.synchronize()
                        times.append(time.perf_counter() - t0)
                fwd_ms = np.median(times) * 1000
                tok_s = B * T / (fwd_ms / 1000)
                print('  T={:>5}, B={}: {:>8.2f} ms ({:>8.0f} tok/s)'.format(T, B, fwd_ms, tok_s))
                results[(name, T, B)] = fwd_ms
            except Exception as e:
                print('  T={:>5}, B={}: {}'.format(T, B, str(e)[:60]))

bench_one('GPT 3L/128D', lambda T: ngpt.GPT(ngpt.GPTConfig(vocab_size=256, n_embd=128, n_layer=3, n_head=4, block_size=T, dropout=0.0, bias=False)).to(device).eval())
bench_one('GPT 8L/192D', lambda T: ngpt.GPT(ngpt.GPTConfig(vocab_size=256, n_embd=192, n_layer=8, n_head=6, block_size=T, dropout=0.0, bias=False)).to(device).eval())
bench_one('Stream 4L/128D', lambda T: Stream(StreamConfig(n_embd=128, n_layer=4, ssm_d_state=8, block_size=T)).to(device).eval())
bench_one('Stream 6L/256D', lambda T: Stream(StreamConfig(n_embd=256, n_layer=6, ssm_d_state=16, block_size=T)).to(device).eval())
bench_one('Stream 4L/128D (compiled)', lambda T: Stream(StreamConfig(n_embd=128, n_layer=4, ssm_d_state=8, block_size=T)).to(device).eval(), use_compile=True)
bench_one('Stream 6L/256D (compiled)', lambda T: Stream(StreamConfig(n_embd=256, n_layer=6, ssm_d_state=16, block_size=T)).to(device).eval(), use_compile=True)

print('\n' + '=' * 100)
h = '{:>5} {:>2} {:>14} {:>14} {:>14} {:>14} {:>14} {:>14}'
print(h.format('T', 'B', 'GPT3_ms', 'GPT8_ms', 'S4_ms', 'S4c_ms', 'S6_ms', 'S6c_ms'))
print('-' * 100)
for T in TS:
    for B in BS:
        def r(n):
            return results.get((n, T, B), None)
        names = ['GPT 3L/128D', 'GPT 8L/192D', 'Stream 4L/128D', 'Stream 4L/128D (compiled)', 'Stream 6L/256D', 'Stream 6L/256D (compiled)']
        vals = [r(n) for n in names]
        if any(v is not None for v in vals):
            line = '{:>5} {:>2}'.format(T, B)
            for v in vals:
                line += '{:>14.2f}'.format(v) if v is not None else '{:>14}'.format('N/A')
            print(line)

import pickle
with open(os.path.join(DRIVE_DIR, 'gpu_bench_results.pkl'), 'wb') as f:
    pickle.dump(results, f)
print('\nResults saved to {}/gpu_bench_results.pkl'.format(DRIVE_DIR))


In [ ]:
# @title 6. Scaling Curve (4 sizes)
import sys, time, numpy as np, torch
sys.path.insert(0, PROJECT_DIR + '/VECTOR')

device = 'cuda'
T = 256
batch_size = 16

# Load data
data = np.memmap('VECTOR/data/bytes/train.bin', dtype=np.uint8, mode='r')

configs = [
    ('tiny',   dict(n_embd=64,  n_layer=2, ssm_d_state=4)),
    ('small',  dict(n_embd=128, n_layer=4, ssm_d_state=8)),
    ('medium', dict(n_embd=192, n_layer=6, ssm_d_state=12)),
    ('large',  dict(n_embd=256, n_layer=8, ssm_d_state=16)),
]

from model import Stream, StreamConfig

scale_results = {}
for name, cfg in configs:
    print(f'\n--- {name} (n_embd={cfg["n_embd"]}, n_layer={cfg["n_layer"]}) ---')
    model = Stream(StreamConfig(n_predict=4, block_size=T, **cfg)).to(device)
    model.train()
    params = sum(p.numel() for p in model.parameters())
    print(f'  Parameters: {params/1e6:.2f}M')

    # Measure tokens/sec
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
    ix = torch.randint(len(data) - T, (batch_size,))
    x = torch.stack([torch.from_numpy(data[i:i+T].astype(np.int64)) for i in ix]).to(device)
    y = x.clone()

    # Warmup
    for _ in range(10):
        _, loss = model(x, targets=y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    # Measured
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(50):
        _, loss = model(x, targets=y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    tok_s = batch_size * T * 50 / elapsed

    print(f'  Throughput: {tok_s:.0f} tokens/sec')
    scale_results[name] = {'params': params, 'tok_s': tok_s, 'val_loss': loss.item()}

print('\n=== Scaling Summary ===')
for name, r in scale_results.items():
    print(f'{name:<10}: {r["params"]/1e6:.2f}M params, {r["tok_s"]:.0f} tok/s, val_loss={r["val_loss"]:.4f}')

with open(os.path.join(DRIVE_DIR, 'scaling_results.pkl'), 'wb') as f:
    pickle.dump(scale_results, f)
print(f'Results saved to {DRIVE_DIR}/scaling_results.pkl')

In [ ]:
# @title 7. Training (configurable)
import sys, os
sys.path.insert(0, PROJECT_DIR + '/VECTOR')

# Pick a config
CONFIG = 'stream_gpu_4k'  # or stream_gpu, stream_light, etc.

print(f'Running config: {CONFIG}')
print(f'Command: python train.py config/{CONFIG}.py')

# The train script uses configurator.py which exec's the config file
# We run it as a subprocess to get clean stdout
!cd VECTOR && python train.py config/{CONFIG}.py 2>&1 | tee {DRIVE_DIR}/train_{CONFIG}.log

# Check if checkpoint was saved
import glob
ckpts = glob.glob('VECTOR/out*/ckpt.pt')
for ckpt in ckpts:
    import shutil
    dest = os.path.join(DRIVE_DIR, os.path.basename(os.path.dirname(ckpt)) + '_ckpt.pt')
    shutil.copy(ckpt, dest)
    print(f'Copied {ckpt} â†’ {dest}')

In [ ]:
# @title 7b. Training WITH Triton (inline, independent)
# Self-contained — finds VECTOR dir, loads modules, enables Triton, trains
import sys, os, time, math, pickle, torch, numpy as np, importlib.util

# --- Find paths (same as cell 4b/4c) ---
vec_dir = None
for d in [os.path.join(os.getcwd(), 'VECTOR'), '/content/RETRANS-X/VECTOR']:
    if os.path.isdir(d):
        vec_dir = d; break
if vec_dir is None:
    p = os.path.dirname(os.getcwd())
    if os.path.isdir(os.path.join(p, 'VECTOR')): vec_dir = os.path.join(p, 'VECTOR')
assert vec_dir is not None, 'Cannot find VECTOR/ dir'
proj_dir = os.path.dirname(vec_dir)
print(f'VECTOR dir: {vec_dir}')

# --- Load triton_scan + model ---
ts = importlib.util.module_from_spec(
    (s := importlib.util.spec_from_file_location('ts', os.path.join(vec_dir, 'triton_scan.py')))
); s.loader.exec_module(ts)
md = importlib.util.module_from_spec(
    (s := importlib.util.spec_from_file_location('md', os.path.join(vec_dir, 'model.py')))
); s.loader.exec_module(md)

# --- Config ---
cfg = dict(
    n_embd=128, n_layer=4, ssm_d_state=8, n_predict=4, block_size=4096,
    batch_size=4, max_iters=500,
    lr=6e-4, min_lr=6e-5, warmup=50, weight_decay=1e-1,
    beta1=0.9, beta2=0.95, grad_clip=1.0,
    out_dir='out_stream_triton_4k',
)
device = 'cuda'

# --- Data ---
data_dir = os.path.join(vec_dir, 'data', 'bytes')
train_bin = os.path.join(data_dir, 'train.bin')
val_bin = os.path.join(data_dir, 'val.bin')
if not os.path.isfile(train_bin):
    print('Data not found at', train_bin)
    print('Run cell 3 first to prepare data.')
    raise SystemExit(1)
train_data = np.memmap(train_bin, dtype=np.uint8, mode='r')
val_data = np.memmap(val_bin, dtype=np.uint8, mode='r')
print(f'Train data: {len(train_data)/1e6:.0f} MB, Val data: {len(val_data)/1e6:.0f} MB')

def get_batch(split):
    data = train_data if split == 'train' else val_data
    T = cfg['block_size']
    ix = torch.randint(len(data) - T, (cfg['batch_size'],))
    x = torch.stack([torch.from_numpy(data[i:i+T].astype(np.int64)) for i in ix]).to(device)
    y = torch.stack([torch.from_numpy(data[i+1:i+T+1].astype(np.int64)) for i in ix]).to(device)
    return x, y

# --- Model ---
model = md.Stream(md.StreamConfig(
    n_embd=cfg['n_embd'], n_layer=cfg['n_layer'],
    ssm_d_state=cfg['ssm_d_state'], n_predict=cfg['n_predict'],
    block_size=cfg['block_size'], dropout=0.0, bias=False,
)).to(device)
ts.enable_triton(model, enabled=True)
model.train()

# --- Optimizer ---
def configure_optimizers(model, wd, lr, betas):
    decay = [p for n,p in model.named_parameters() if p.dim() >= 2 and p.requires_grad]
    nodecay = [p for n,p in model.named_parameters() if p.dim() < 2 and p.requires_grad]
    return torch.optim.AdamW([
        {'params': decay, 'weight_decay': wd},
        {'params': nodecay, 'weight_decay': 0.0},
    ], lr=lr, betas=betas, fused=True)

optimizer = configure_optimizers(model, cfg['weight_decay'], cfg['lr'], (cfg['beta1'], cfg['beta2']))

# --- Training loop ---
max_iters = cfg['max_iters']
warmup = cfg['warmup']
log_interval = 10
eval_interval = 250

log = []
t0 = time.perf_counter()

for step in range(max_iters + 1):
    # LR schedule
    if step < warmup:
        lr_i = cfg['lr'] * step / warmup
    else:
        ratio = (step - warmup) / (max_iters - warmup)
        lr_i = cfg['min_lr'] + 0.5 * (cfg['lr'] - cfg['min_lr']) * (1 + math.cos(math.pi * ratio))
    for pg in optimizer.param_groups:
        pg['lr'] = lr_i

    x, y = get_batch('train')
    _, loss = model(x, targets=y)
    loss.backward()

    if cfg['grad_clip'] > 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['grad_clip'])

    optimizer.step()
    optimizer.zero_grad()

    dt = time.perf_counter() - t0
    log.append(dict(step=step, loss=loss.item(), lr=lr_i, ms=dt*1000))
    t0 = time.perf_counter()

    if step % log_interval == 0:
        print(f'iter {step}: loss {loss.item():.4f}, lr {lr_i:.2e}, time {dt*1000:.1f}ms')

    if step > 0 and step % eval_interval == 0:
        model.eval()
        with torch.no_grad():
            xv, yv = get_batch('val')
            _, val_loss = model(xv, targets=yv)
        print(f'step {step}: train loss {loss.item():.4f}, val loss {val_loss.item():.4f}')
        model.train()

# --- Final eval ---
model.eval()
with torch.no_grad():
    xv, yv = get_batch('val')
    _, final_val = model(xv, targets=yv)
print(f'\nFinal val loss: {final_val.item():.4f}')

# --- Save ---
out = os.path.join(vec_dir, cfg['out_dir'])
os.makedirs(out, exist_ok=True)
ckpt = {'model': model.state_dict(), 'config': cfg, 'log': log, 'val_loss': final_val.item()}
torch.save(ckpt, os.path.join(out, 'ckpt.pt'))
print(f'Checkpoint saved to {out}/ckpt.pt')

# Copy to Drive
try:
    dr = '/content/drive/MyDrive/stream_results'
    os.makedirs(dr, exist_ok=True)
    torch.save(ckpt, os.path.join(dr, 'triton_train_ckpt.pt'))
    print(f'Copied to {dr}/triton_train_ckpt.pt')
except Exception as e:
    print(f'Drive save skipped: {e}')

# Print speed summary
avg_ms = np.median([l['ms'] for l in log[5:]])  # skip warmup
tok_s = cfg['batch_size'] * cfg['block_size'] / (avg_ms / 1000)
print(f'\nAvg iter: {avg_ms:.0f}ms, Throughput: {tok_s:.0f} tok/s')


In [ ]:
# @title 7c. Triton Training at T=16384 (long sequences)
# Same as 7b but with block_size=16384 — the O(n) advantage shines here
import sys, os, time, math, pickle, torch, numpy as np, importlib.util

# --- Find paths ---
vec_dir = None
for d in [os.path.join(os.getcwd(), 'VECTOR'), '/content/RETRANS-X/VECTOR']:
    if os.path.isdir(d):
        vec_dir = d; break
if vec_dir is None:
    p = os.path.dirname(os.getcwd())
    if os.path.isdir(os.path.join(p, 'VECTOR')): vec_dir = os.path.join(p, 'VECTOR')
assert vec_dir is not None, 'Cannot find VECTOR/ dir'
print(f'VECTOR dir: {vec_dir}')

# --- Load modules ---
ts = importlib.util.module_from_spec(
    (s := importlib.util.spec_from_file_location('ts', os.path.join(vec_dir, 'triton_scan.py')))
); s.loader.exec_module(ts)
md = importlib.util.module_from_spec(
    (s := importlib.util.spec_from_file_location('md', os.path.join(vec_dir, 'model.py')))
); s.loader.exec_module(md)

# --- Config: T=16384 (4x longer), B=2 (memory safe) ---
cfg = dict(
    n_embd=128, n_layer=4, ssm_d_state=8, n_predict=4, block_size=16384,
    batch_size=2, max_iters=500,
    lr=6e-4, min_lr=6e-5, warmup=50, weight_decay=1e-1,
    beta1=0.9, beta2=0.95, grad_clip=1.0,
    out_dir='out_stream_triton_16k',
)
device = 'cuda'

# --- Data ---
data_dir = os.path.join(vec_dir, 'data', 'bytes')
train_bin = os.path.join(data_dir, 'train.bin')
val_bin = os.path.join(data_dir, 'val.bin')
if not os.path.isfile(train_bin):
    print('Data not found at', train_bin); raise SystemExit(1)
train_data = np.memmap(train_bin, dtype=np.uint8, mode='r')
val_data = np.memmap(val_bin, dtype=np.uint8, mode='r')
print(f'Train data: {len(train_data)/1e6:.0f} MB, Val data: {len(val_data)/1e6:.0f} MB')
print(f'Effective samples at T={cfg["block_size"]}: {len(train_data) // cfg["block_size"]:,}')

def get_batch(split):
    data = train_data if split == 'train' else val_data
    T = cfg['block_size']
    ix = torch.randint(len(data) - T, (cfg['batch_size'],))
    x = torch.stack([torch.from_numpy(data[i:i+T].astype(np.int64)) for i in ix]).to(device)
    y = torch.stack([torch.from_numpy(data[i+1:i+T+1].astype(np.int64)) for i in ix]).to(device)
    return x, y

# --- Model ---
model = md.Stream(md.StreamConfig(
    n_embd=cfg['n_embd'], n_layer=cfg['n_layer'],
    ssm_d_state=cfg['ssm_d_state'], n_predict=cfg['n_predict'],
    block_size=cfg['block_size'], dropout=0.0, bias=False,
)).to(device)
ts.enable_triton(model, enabled=True)
model.train()

# --- Optimizer ---
def configure_optimizers(model, wd, lr, betas):
    decay = [p for n,p in model.named_parameters() if p.dim() >= 2 and p.requires_grad]
    nodecay = [p for n,p in model.named_parameters() if p.dim() < 2 and p.requires_grad]
    return torch.optim.AdamW([
        {'params': decay, 'weight_decay': wd},
        {'params': nodecay, 'weight_decay': 0.0},
    ], lr=lr, betas=betas, fused=True)

optimizer = configure_optimizers(model, cfg['weight_decay'], cfg['lr'], (cfg['beta1'], cfg['beta2']))

# --- Training loop ---
max_iters = cfg['max_iters']
warmup = cfg['warmup']
log_interval = 10
eval_interval = 250

log = []
t0 = time.perf_counter()

for step in range(max_iters + 1):
    if step < warmup:
        lr_i = cfg['lr'] * step / warmup
    else:
        ratio = (step - warmup) / (max_iters - warmup)
        lr_i = cfg['min_lr'] + 0.5 * (cfg['lr'] - cfg['min_lr']) * (1 + math.cos(math.pi * ratio))
    for pg in optimizer.param_groups:
        pg['lr'] = lr_i

    x, y = get_batch('train')
    _, loss = model(x, targets=y)
    loss.backward()

    if cfg['grad_clip'] > 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['grad_clip'])

    optimizer.step()
    optimizer.zero_grad()

    dt = time.perf_counter() - t0
    log.append(dict(step=step, loss=loss.item(), lr=lr_i, ms=dt*1000))
    t0 = time.perf_counter()

    if step % log_interval == 0:
        print(f'iter {step}: loss {loss.item():.4f}, lr {lr_i:.2e}, time {dt*1000:.1f}ms')

    if step > 0 and step % eval_interval == 0:
        model.eval()
        with torch.no_grad():
            xv, yv = get_batch('val')
            _, val_loss = model(xv, targets=yv)
        print(f'step {step}: train loss {loss.item():.4f}, val loss {val_loss.item():.4f}')
        model.train()

# --- Final eval ---
model.eval()
with torch.no_grad():
    xv, yv = get_batch('val')
    _, final_val = model(xv, targets=yv)
print(f'\nFinal val loss: {final_val.item():.4f}')

# --- Save ---
out = os.path.join(vec_dir, cfg['out_dir'])
os.makedirs(out, exist_ok=True)
ckpt = {'model': model.state_dict(), 'config': cfg, 'log': log, 'val_loss': final_val.item()}
torch.save(ckpt, os.path.join(out, 'ckpt.pt'))
print(f'Checkpoint saved to {out}/ckpt.pt')

try:
    dr = '/content/drive/MyDrive/stream_results'
    os.makedirs(dr, exist_ok=True)
    torch.save(ckpt, os.path.join(dr, 'triton_train_16k_ckpt.pt'))
    print(f'Copied to {dr}/triton_train_16k_ckpt.pt')
except Exception as e:
    print(f'Drive save skipped: {e}')

avg_ms = np.median([l['ms'] for l in log[5:]])
tok_s = cfg['batch_size'] * cfg['block_size'] / (avg_ms / 1000)
print(f'\nT={cfg["block_size"]}, B={cfg["batch_size"]}')
print(f'Avg iter: {avg_ms:.0f}ms, Throughput: {tok_s:.0f} tok/s')
print(f'Tokens per iter: {cfg["batch_size"] * cfg["block_size"]}')


In [ ]:
# @title 7d. Memory Footprint: O(n) Scaling (Stream 4L/128D)
# Measures peak CUDA memory at T = 4096, 8192, 16384, 32768, 65536
# Run after cell 4b (or 7b). Self-contained path finding.
import sys, os, time, torch, numpy as np, importlib.util, gc

vec_dir = None
for d in [os.path.join(os.getcwd(), 'VECTOR'), '/content/RETRANS-X/VECTOR']:
    if os.path.isdir(d): vec_dir = d; break
if vec_dir is None:
    p = os.path.dirname(os.getcwd())
    if os.path.isdir(os.path.join(p, 'VECTOR')): vec_dir = os.path.join(p, 'VECTOR')
assert vec_dir is not None

ts = importlib.util.module_from_spec(
    (s := importlib.util.spec_from_file_location('ts', os.path.join(vec_dir, 'triton_scan.py')))
); s.loader.exec_module(ts)
md = importlib.util.module_from_spec(
    (s := importlib.util.spec_from_file_location('md', os.path.join(vec_dir, 'model.py')))
); s.loader.exec_module(md)

device = 'cuda'
TS = [4096, 8192, 16384, 32768, 65536]

def measure_mem(T, B=1):
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    model = md.Stream(md.StreamConfig(
        n_embd=128, n_layer=4, ssm_d_state=8, block_size=T,
    )).to(device)
    ts.enable_triton(model, True)
    model.train()
    x = torch.randint(0, 256, (B, T), device=device)
    _, loss = model(x, targets=x)
    loss.backward()
    peak = torch.cuda.max_memory_allocated(device)
    return peak, peak / 1e9

print(f"{'T':>7} {'B':>2} {'Peak VRAM':>12} {'Scaling':>10}")
print('-' * 35)
ref = None
for T in TS:
    try:
        b, gb = measure_mem(T)
        ratio = 1.0 if ref is None else b / ref
        mult = T / 4096
        ideal = 1.0 if ref is None else mult
        eff = ratio / ideal if ref else 1.0
        print(f'{T:>7} {1:>2} {gb:>8.2f} GB {ratio:>8.2f}x (O(n) ideal: {ideal:.1f}x)')
        if ref is None: ref = b
    except RuntimeError as e:
        if 'out of memory' in str(e).lower():
            print(f'{T:>7} {1:>2}     OOM')
            torch.cuda.empty_cache()
        else:
            raise

print(f'\nGPT comparison: at T=16384 flash-attn GPT uses ~3-5 GB (similar-scale).')
print(f'At T=32768 and beyond, GPT flash-attn O(n^2) compute explodes even if memory is tiled.')


In [ ]:
# @title 7e. Long Triton Training: 5000 iters (Stream 4L/128D)
# Self-contained. Runs 5000 iters at T=4096, B=4, saves checkpoints every 500 iters.
import sys, os, time, math, pickle, torch, numpy as np, importlib.util, json

vec_dir = None
for d in [os.path.join(os.getcwd(), 'VECTOR'), '/content/RETRANS-X/VECTOR']:
    if os.path.isdir(d): vec_dir = d; break
if vec_dir is None:
    p = os.path.dirname(os.getcwd())
    if os.path.isdir(os.path.join(p, 'VECTOR')): vec_dir = os.path.join(p, 'VECTOR')
assert vec_dir is not None, 'Cannot find VECTOR/ dir'
print(f'VECTOR dir: {vec_dir}')

ts = importlib.util.module_from_spec(
    (s := importlib.util.spec_from_file_location('ts', os.path.join(vec_dir, 'triton_scan.py')))
); s.loader.exec_module(ts)
md = importlib.util.module_from_spec(
    (s := importlib.util.spec_from_file_location('md', os.path.join(vec_dir, 'model.py')))
); s.loader.exec_module(md)

cfg = dict(n_embd=128, n_layer=4, ssm_d_state=8, n_predict=4, block_size=4096,
           batch_size=4, max_iters=5000, lr=6e-4, min_lr=6e-5, warmup_iters=200,
           weight_decay=1e-1, beta1=0.9, beta2=0.95, grad_clip=1.0,
           out_dir='out_stream_long')
device = 'cuda'

data_dir = os.path.join(vec_dir, 'data', 'bytes')
train_data = np.memmap(os.path.join(data_dir, 'train.bin'), dtype=np.uint8, mode='r')
val_data = np.memmap(os.path.join(data_dir, 'val.bin'), dtype=np.uint8, mode='r')

def get_batch(split):
    data = train_data if split == 'train' else val_data
    T = cfg['block_size']
    ix = torch.randint(len(data) - T - 1, (cfg['batch_size'],))
    x = torch.stack([torch.from_numpy(data[i:i+T].astype(np.int64)) for i in ix]).to(device)
    y = torch.stack([torch.from_numpy(data[i+1:i+T+1].astype(np.int64)) for i in ix]).to(device)
    return x, y

model = md.Stream(md.StreamConfig(
    n_embd=cfg['n_embd'], n_layer=cfg['n_layer'],
    ssm_d_state=cfg['ssm_d_state'], n_predict=cfg['n_predict'],
    block_size=cfg['block_size'], dropout=0.0, bias=False,
)).to(device)
ts.enable_triton(model, enabled=True)
model.train()

def configure_optimizers(model, wd, lr, betas):
    decay = [p for n,p in model.named_parameters() if p.dim() >= 2 and p.requires_grad]
    nodecay = [p for n,p in model.named_parameters() if p.dim() < 2 and p.requires_grad]
    return torch.optim.AdamW([
        {'params': decay, 'weight_decay': wd},
        {'params': nodecay, 'weight_decay': 0.0},
    ], lr=lr, betas=betas, fused=True)

optimizer = configure_optimizers(model, cfg['weight_decay'], cfg['lr'], (cfg['beta1'], cfg['beta2']))

log = []
t0 = time.perf_counter()
checkpoint_dir = os.path.join(vec_dir, cfg['out_dir'])
os.makedirs(checkpoint_dir, exist_ok=True)

for step in range(cfg['max_iters'] + 1):
    if step < cfg['warmup_iters']:
        lr_i = cfg['lr'] * step / cfg['warmup_iters']
    else:
        progress = (step - cfg['warmup_iters']) / (cfg['max_iters'] - cfg['warmup_iters'])
        lr_i = cfg['min_lr'] + 0.5 * (cfg['lr'] - cfg['min_lr']) * (1 + math.cos(math.pi * progress))
    for pg in optimizer.param_groups:
        pg['lr'] = lr_i

    x, y = get_batch('train')
    _, loss = model(x, targets=y)
    loss.backward()
    if cfg['grad_clip'] > 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['grad_clip'])
    optimizer.step()
    optimizer.zero_grad()

    dt = time.perf_counter() - t0
    log.append(dict(step=step, train_loss=loss.item(), lr=lr_i, ms=dt*1000))
    t0 = time.perf_counter()

    if step % 50 == 0:
        print(f'iter {step:>5}: loss {loss.item():.4f}, lr {lr_i:.2e}, {dt*1000:.0f}ms')

    if step > 0 and step % 250 == 0:
        model.eval()
        with torch.no_grad():
            xv, yv = get_batch('val')
            _, val_loss = model(xv, targets=yv)
        print(f'  EVAL iter {step}: train {loss.item():.4f}, val {val_loss.item():.4f}')
        log[-1]['val_loss'] = val_loss.item()
        model.train()

    if step > 0 and step % 500 == 0:
        ckpt = {'model': model.state_dict(), 'config': cfg, 'log': log[-500:]}
        torch.save(ckpt, os.path.join(checkpoint_dir, f'ckpt_{step}.pt'))
        print(f'  Checkpoint saved: ckpt_{step}.pt')

model.eval()
with torch.no_grad():
    xv, yv = get_batch('val')
    _, final_val = model(xv, targets=yv)
print(f'\nFinal val loss: {final_val.item():.4f}')

full_log = {'config': cfg, 'log': log, 'final_val_loss': final_val.item()}
torch.save(full_log, os.path.join(checkpoint_dir, 'full_log.pt'))
torch.save(model.state_dict(), os.path.join(checkpoint_dir, 'ckpt_final.pt'))
print(f'Full log + final checkpoint saved to {checkpoint_dir}/')

try:
    dr = '/content/drive/MyDrive/stream_results'
    os.makedirs(dr, exist_ok=True)
    torch.save(full_log, os.path.join(dr, 'long_train_log.pt'))
    torch.save(model.state_dict(), os.path.join(dr, 'long_train_final.pt'))
    print(f'Copied to Drive: {dr}/')
except Exception as e:
    print(f'Drive save skipped: {e}')

avg_ms = np.median([l['ms'] for l in log[10:]])
tok_s = cfg['batch_size'] * cfg['block_size'] / (avg_ms / 1000)
print(f'\nAvg iter: {avg_ms:.0f}ms, Throughput: {tok_s:.0f} tok/s')
print(f'Total: {len(log)} iters in {sum(l["ms"] for l in log)/1000/60:.1f} min')

# Quick summary
val_pts = [l for l in log if 'val_loss' in l]
if val_pts:
    print(f'\nVal loss trajectory:')
    for l in val_pts[::max(1, len(val_pts)//10)]:
        print(f'  iter {l["step"]:>5}: val {l["val_loss"]:.4f}')
    print(f'  iter {val_pts[-1]["step"]:>5}: val {val_pts[-1]["val_loss"]:.4f} (final)')


In [ ]:
# @title 8. Download Results to Local Machine
import os

DRIVE_DIR = '/content/drive/MyDrive/stream_results'

print(f'Results are in: {DRIVE_DIR}')
print('\nFiles:')
for f in os.listdir(DRIVE_DIR):
    size = os.path.getsize(os.path.join(DRIVE_DIR, f))
    print(f'  {f}  ({size/1e3:.0f} KB)')

# zip for easy download
import shutil
shutil.make_archive('/content/stream_results', 'zip', DRIVE_DIR)
print('\nResults also zipped to: /content/stream_results.zip')
print('Download via: Files sidebar â†’ stream_results.zip â†’ Download')

In [ ]:
# @title 9. GPT 3L/128D Baseline Training (T=4096, 5000 iters)
# Trains nanoGPT on the same data/config as Stream cell 7e for direct comparison
import sys, os, time, math, torch, numpy as np, importlib.util

vec_dir = None
for d in [os.path.join(os.getcwd(), 'VECTOR'), '/content/RETRANS-X/VECTOR']:
    if os.path.isdir(d): vec_dir = d; break
if vec_dir is None:
    p = os.path.dirname(os.getcwd())
    if os.path.isdir(os.path.join(p, 'VECTOR')): vec_dir = os.path.join(p, 'VECTOR')
assert vec_dir is not None
proj_dir = os.path.dirname(vec_dir)

# --- Load nanoGPT ---
ng_spec = importlib.util.spec_from_file_location('ngpt', os.path.join(proj_dir, 'nanoGPT', 'model.py'))
ng_mod = importlib.util.module_from_spec(ng_spec); ng_spec.loader.exec_module(ng_mod)
GPT = ng_mod.GPT; GPTConfig = ng_mod.GPTConfig

cfg = dict(vocab_size=256, n_embd=128, n_layer=3, n_head=4, block_size=4096,
           batch_size=4, max_iters=5000, lr=6e-4, min_lr=6e-5, warmup_iters=200,
           weight_decay=1e-1, beta1=0.9, beta2=0.95, grad_clip=1.0,
           dropout=0.0, bias=False, out_dir='out_gpt_baseline')
device = 'cuda'

data_dir = os.path.join(vec_dir, 'data', 'bytes')
train_data = np.memmap(os.path.join(data_dir, 'train.bin'), dtype=np.uint8, mode='r')
val_data = np.memmap(os.path.join(data_dir, 'val.bin'), dtype=np.uint8, mode='r')

def get_batch(split):
    data = train_data if split == 'train' else val_data
    T = cfg['block_size']
    ix = torch.randint(len(data) - T - 1, (cfg['batch_size'],))
    x = torch.stack([torch.from_numpy(data[i:i+T].astype(np.int64)) for i in ix]).to(device)
    y = torch.stack([torch.from_numpy(data[i+1:i+T+1].astype(np.int64)) for i in ix]).to(device)
    return x, y

model = GPT(GPTConfig(**cfg)).to(device).train()
print(f'GPT params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M')

decay = [p for n,p in model.named_parameters() if p.dim() >= 2 and p.requires_grad]
nodecay = [p for n,p in model.named_parameters() if p.dim() < 2 and p.requires_grad]
optimizer = torch.optim.AdamW([
    {'params': decay, 'weight_decay': cfg['weight_decay']},
    {'params': nodecay, 'weight_decay': 0.0},
], lr=cfg['lr'], betas=(cfg['beta1'], cfg['beta2']), fused=True)

log = []
t0 = time.perf_counter()
checkpoint_dir = os.path.join(proj_dir, cfg['out_dir'])
os.makedirs(checkpoint_dir, exist_ok=True)

for step in range(cfg['max_iters'] + 1):
    if step < cfg['warmup_iters']:
        lr_i = cfg['lr'] * step / cfg['warmup_iters']
    else:
        progress = (step - cfg['warmup_iters']) / (cfg['max_iters'] - cfg['warmup_iters'])
        lr_i = cfg['min_lr'] + 0.5 * (cfg['lr'] - cfg['min_lr']) * (1 + math.cos(math.pi * progress))
    for pg in optimizer.param_groups:
        pg['lr'] = lr_i

    x, y = get_batch('train')
    logits, loss = model(x, targets=y)
    loss.backward()
    if cfg['grad_clip'] > 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['grad_clip'])
    optimizer.step()
    optimizer.zero_grad()

    dt = time.perf_counter() - t0
    log.append(dict(step=step, train_loss=loss.item(), lr=lr_i, ms=dt*1000))
    t0 = time.perf_counter()

    if step % 50 == 0:
        print(f'iter {step:>5}: loss {loss.item():.4f}, lr {lr_i:.2e}, {dt*1000:.0f}ms')

    if step > 0 and step % 250 == 0:
        model.eval()
        with torch.no_grad():
            xv, yv = get_batch('val')
            _, val_loss = model(xv, targets=yv)
        print(f'  EVAL iter {step}: train {loss.item():.4f}, val {val_loss.item():.4f}')
        log[-1]['val_loss'] = val_loss.item()
        model.train()

    if step > 0 and step % 500 == 0:
        torch.save({'model': model.state_dict(), 'config': cfg, 'log': log[-500:]},
                   os.path.join(checkpoint_dir, f'ckpt_{step}.pt'))
        print(f'  Checkpoint: ckpt_{step}.pt')

model.eval()
with torch.no_grad():
    xv, yv = get_batch('val')
    _, final_val = model(xv, targets=yv)
print(f'\nGPT Final val loss: {final_val.item():.4f}')

full_log = {'config': cfg, 'log': log, 'final_val_loss': final_val.item()}
torch.save(full_log, os.path.join(checkpoint_dir, 'full_log.pt'))
torch.save(model.state_dict(), os.path.join(checkpoint_dir, 'ckpt_final.pt'))

try:
    dr = '/content/drive/MyDrive/stream_results'
    os.makedirs(dr, exist_ok=True)
    torch.save(full_log, os.path.join(dr, 'gpt_baseline_log.pt'))
    torch.save(model.state_dict(), os.path.join(dr, 'gpt_baseline_final.pt'))
    print(f'Copied to Drive: {dr}/')
except Exception as e:
    print(f'Drive save skipped: {e}')

val_pts = [l for l in log if 'val_loss' in l]
if val_pts:
    print(f'\nGPT val loss trajectory:')
    for l in val_pts[::max(1, len(val_pts)//10)]:
        print(f'  iter {l["step"]:>5}: val {l["val_loss"]:.4f}')
    print(f'  iter {val_pts[-1]["step"]:>5}: val {val_pts[-1]["val_loss"]:.4f} (final)')
